# Lecture 3 - Excercises

Problem 3.1. Optimise the capacity and dispatch of solar PV, onshore wind, and open-cycle gas turbine (OCGT) generators to supply the inelastic electricity demand throughout one year.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Read csv files for PV and onshore wind capacity factors and demand
pv = pd.read_csv("pv_optimal.csv", sep=";")
wind = pd.read_csv("onshore_wind.csv", sep=";")
demand = pd.read_csv("electricity_demand.csv", sep=";")

# Set time as index and convert to datetime
pv["utc_time"] = pd.to_datetime(pv["utc_time"])
wind["utc_time"] = pd.to_datetime(wind["utc_time"])
demand["utc_time"] = pd.to_datetime(demand["utc_time"])

# Use data from Portugal in 2015
pv = pv[pv["utc_time"].dt.year == 2015]
pv = pv[["utc_time","PRT"]]
wind = wind[wind["utc_time"].dt.year == 2015]
wind = wind[["utc_time","PRT"]]
demand = demand[demand["utc_time"].dt.year == 2015]
demand = demand[["utc_time","PRT"]]

# additional data
tech = ["Onshore wind", "PV", "OCGT"]
cc = [101_644, 51_346, 47_718]
mc = [0, 0, 64.7]

# Efficiency of OCGT
eta_ocgt = 0.41

### 3.1a
Calculate the total system cost, the optimal installed capacities, the annual generation per technology, and plot the hourly generation and demand during January.

In [2]:
import gurobipy as gp
from gurobipy import GRB

# Create new model
m31a = gp.Model()

time = range(len(pv))

# Add decision variables for generation
g_pv = m31a.addVars(time, name="gen_pv", lb = 0)
g_wind = m31a.addVars(time, name="gen_wind", lb = 0)
g_ocgt = m31a.addVars(time, name="gen_ocgt", lb = 0)

c_pv = m31a.addVar(name="cap_pv", lb=0)
c_wind = m31a.addVar(name = "cap_wind", lb = 0)
c_ocgt = m31a.addVar(name = "cap_ocgt", lb = 0)


# Add objective function minimize generation costs 
m31a.setObjective(gp.quicksum(cc[0] * g_wind[t] + cc[1] * g_pv[t] + 
                              cc[2] * g_ocgt[t] for t in time) + gp.quicksum(mc[0] * g_wind[t] + mc[1] * g_pv[t] + 
                              mc[2] * g_ocgt[t] for t in time), GRB.MINIMIZE)

# Add constraints for generation limits
m31a.addConstrs(g_wind[t] <= c_wind * wind.iloc[t, 1] for t in time)
m31a.addConstrs(g_pv[t] <= c_pv * pv.iloc[t, 1] for t in time)
m31a.addConstrs(g_ocgt[t] <= c_ocgt * eta_ocgt for t in time) 

# Add balancing constraint
m31a.addConstrs(gp.quicksum(g_wind[t] + g_pv[t] + g_ocgt[t] for t in time) >= demand.iloc[t,1] for t in time)

# Solve the model
m31a.optimize()

# Display the results
print(f"Total System Cost: {m31a.ObjVal}")



Set parameter Username
Set parameter LicenseID to value 2745305
Academic license - for non-commercial use only - expires 2026-11-26
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 35040 rows, 26283 columns and 230260998 nonzeros (Min)
Model fingerprint: 0x1349b6d9
Model has 26280 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-03, 1e+00]
  Objective range  [5e+04, 1e+05]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+03, 9e+03]

Out of memory


GurobiError: Out of memory